<a href="https://colab.research.google.com/github/ilfpns/HailMary_AI/blob/main/HM_yolo26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import importlib.util
import subprocess
import sys
import os
import shutil

if importlib.util.find_spec("ultralytics") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics"])

import torch
from google.colab import drive
from ultralytics import YOLO
from pathlib import Path
from PIL import Image, ImageDraw
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [ ]:
drive.mount("/content/drive")
src = Path("/content/drive/MyDrive/HM_img2/HM_img2.zip")
assert src.exists(), f"zip이 없습니다: {src}"
print("사용할 zip:", src.name, f"{src.stat().st_size / 1024**3:.2f} GB")

extract = Path("/content/data")
if not any(extract.rglob("data.yaml")):
    shutil.copy(src, "/content/dataset.zip")
    extract.mkdir(exist_ok=True)
    subprocess.run(["unzip", "-q", "-o", "/content/dataset.zip", "-d", str(extract)], check=True)
    Path("/content/dataset.zip").unlink()

root = next(extract.rglob("data.yaml")).parent
print("root:", root)

yaml_path = root / "data.yaml"
lines = [f"path: {root}" if l.startswith("path:") else l for l in yaml_path.read_text().splitlines()]
if not any(l.startswith("path:") for l in lines):
    lines.insert(0, f"path: {root}")
yaml_path.write_text("\n".join(lines) + "\n")

for s in ["train", "valid", "test"]:
    if (root / s).exists():
        print(s, "이미지", len(list((root / s / "images").iterdir())), "라벨", len(list((root / s / "labels").iterdir())))

Mounted at /content/drive
사용할 zip: HM_img2.zip 22.24 GB


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "ultralytics"], check=True)

drive.mount("/content/drive")

ZIP = Path("/content/drive/MyDrive/HM_img2/HM_img2.zip")
PROJECT = Path("/content/drive/MyDrive/HM_img2/runs")
NAME = "HM_yolo26n"
MODEL = "yolo26n.pt"
EPOCHS = 100

extract = Path("/content/data")
if not any(extract.rglob("data.yaml")):
    shutil.copy(ZIP, "/content/dataset.zip")
    extract.mkdir(exist_ok=True)
    subprocess.run(["unzip", "-q", "-o", "/content/dataset.zip", "-d", str(extract)], check=True)
    Path("/content/dataset.zip").unlink()

root = next(extract.rglob("data.yaml")).parent
yaml_path = root / "data.yaml"
lines = [l for l in yaml_path.read_text().splitlines() if not l.startswith("path:")]
yaml_path.write_text("\n".join([f"path: {root}"] + lines) + "\n")

weights = PROJECT / NAME / "weights"

def load_epoch(pt):
    try:
        return torch.load(pt, map_location="cpu", weights_only=False).get("epoch", -1)
    except Exception as e:
        print("체크포인트 읽기 실패:", pt.name, e)
        return None

def find_checkpoint():
    if not weights.exists():
        return None, None
    periodic = sorted(weights.glob("epoch*.pt"), key=lambda p: int(re.findall(r"\d+", p.stem)[0]), reverse=True)
    for pt in [weights / "last.pt", *periodic]:
        if pt.exists():
            ep = load_epoch(pt)
            if ep is not None:
                return pt, ep
    return None, None

ckpt, ep = find_checkpoint()

if ckpt is not None and ep == -1:
    print("이미 학습이 끝난 run입니다:", PROJECT / NAME)
elif ckpt is not None:
    print(f"이어서 학습: {ckpt.name} (완료된 epoch {ep + 1}/{EPOCHS})")
    YOLO(str(ckpt)).train(resume=True)
else:
    print("처음부터 학습")
    YOLO(MODEL).train(
        data=str(yaml_path),
        epochs=EPOCHS,
        patience=30,
        imgsz=640,
        batch=32,
        workers=2,
        device=0,
        project=str(PROJECT),
        name=NAME,
        exist_ok=True,
        save_period=2,
        seed=42,
    )

best = YOLO(str(weights / "best.pt"))
m = best.val(data=str(yaml_path), split="test", imgsz=640)
for j, c in enumerate(m.box.ap_class_index):
    print(f"{best.names[int(c)]:<12} mAP50={m.box.ap50[j]:.3f}  mAP50-95={m.box.ap[j]:.3f}")